In [5]:
# XML Parsing

import xml.etree.ElementTree as ET

ssml_data = ""

with open("sentence_01.xml","r", encoding="utf-8") as file:
    ssml_data = file.read()

root = ET.fromstring(ssml_data)

print(f"Root tag: {root.tag}") 

for paragraph in root.findall('p'):
    for sentence in paragraph.findall('s'):
        print(f"Found sentence text: {''.join(sentence.itertext())}")


Root tag: speak
Found sentence text: यो कौण है?
Found sentence text: यो कौण है?


In [13]:
# Structure Analysis

import nltk

nltk.download('punkt_tab') 
from nltk.tokenize import sent_tokenize
with open("raw_texr.txt","r", encoding="utf-8") as file:
    raw_text = file.read() 
sentences = sent_tokenize(raw_text)

for index, sentence in enumerate(sentences):
    print(f"Sentence {index + 1}: {sentence}")

Sentence 1: मैं ईभी घरू जाण लगरा। घरू जा के बात करूमा। यो कौण है?


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [14]:
# Structure Analysis

import re

with open("raw_texr.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

sentences = re.split(r'[।?!]+', raw_text)

sentences = [s.strip() for s in sentences if s.strip()]

for index, sentence in enumerate(sentences):
    print(f"Sentence {index + 1}: {sentence}")

Sentence 1: मैं ईभी घरू जाण लगरा
Sentence 2: घरू जा के बात करूमा
Sentence 3: यो कौण है


In [23]:
import re
from num2words import num2words

def process_sub_tags(xml_tree):
    for sub_tag in xml_tree.findall('.//sub'):
        alias_text = sub_tag.get('alias')
        if alias_text:
            sub_tag.text = alias_text
    return xml_tree

def automated_text_normalization(raw_text):
    
    def currency_repl(match):
        amount = int(match.group(1))
        words = num2words(amount, lang='hi')
        return f"{words} रूपये"
    
    text = re.sub(r'₹(\d+)', currency_repl, raw_text)
    
    def number_repl(match):
        number = int(match.group(0))
        return num2words(number, lang='hi')
    
    text = re.sub(r'\b\d+\b', number_repl, text)
    
    return text

ssml_text = "मेरे धोरे ₹200 है।"

normalized_text = automated_text_normalization(ssml_text)
print(f"Normalized Text: {normalized_text}")


NotImplementedError: 

In [29]:
import re
from num2words import num2words

def automated_text_normalization(raw_text):

    # Convert currency
    def currency_repl(match):
        amount = int(match.group(1))
        words = num2words(amount, lang='hi')
        return f"{words} रुपये"

    # Convert ₹200
    text = re.sub(r'₹(\d+)', currency_repl, raw_text)

    # Convert remaining numbers
    def number_repl(match):
        number = int(match.group(0))
        return num2words(number, lang='hi')

    text = re.sub(r'\b\d+\b', number_repl, text)

    return text


ssml_text = "मेरे धोरे ₹200 है।"

normalized_text = automated_text_normalization(ssml_text)

print(f"Normalized Text: {normalized_text}")

NotImplementedError: 

In [ ]:
import xml.etree.ElementTree as ET
from g2p_en import G2p

# Initialize the G2P engine (which contains both a dictionary and fallback rules)
g2p = G2p()

def text_to_phonemes(xml_string):
    root = ET.fromstring(xml_string)
    phoneme_output = []
    
    # Iterate through the elements
    for element in root.iter():
        # Path 1: Explicit <phoneme> markup
        if element.tag == 'phoneme':
            # Grab the exact phonemes provided by the author
            explicit_ph = element.attrib.get('ph')
            if explicit_ph:
                phoneme_output.append(f"[{explicit_ph}]")
                
        # Path 2: Automated fallback for standard text
        # (Assuming the text is normalized and not inside a phoneme tag)
        elif element.text and element.text.strip() and element.tag != 'phoneme':
            words = element.text.strip().split()
            for word in words:
                # Ask the G2P engine to guess the phonemes
                phonemes = g2p(word)
                # g2p returns a list of phonemes, let's join them
                phoneme_output.append(" ".join(phonemes))
                
    return " | ".join(phoneme_output)

# --- Pipeline Execution ---
# Notice we have a normal word and a hard-to-pronounce name marked up explicitly
ssml_data = """<?xml version="1.0"?>
<speak>
  Hello 
  <phoneme alphabet="ipa" ph="ˈwʊk.i">Wookiee</phoneme>
</speak>
"""

phoneme_sequence = text_to_phonemes(ssml_data)
print(f"Phoneme Sequence: {phoneme_sequence}")

# OUTPUT:
# Phoneme Sequence: HH AH0 L OW1 | [ˈwʊk.i]

In [1]:
from google.cloud import texttospeech

# 1. Read your XML file
with open("sentence_01.ssml", "r") as file:
    ssml_text = file.read()

client = texttospeech.TextToSpeechClient()
synthesis_input = texttospeech.SynthesisInput(ssml=ssml_text)
voice = texttospeech.VoiceSelectionParams(language_code="hi-IN", name="en-US-Journey-F")
audio_config = texttospeech.AudioConfig(audio_encoding=texttospeech.AudioEncoding.MP3)

# 2. Send to the API
response = client.synthesize_speech(input=synthesis_input, voice=voice, audio_config=audio_config)

# 3. Save the resulting audio
with open("output.mp3", "wb") as out:
    out.write(response.audio_content)

UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 313: character maps to <undefined>

In [3]:
def ssml_to_audio(ssml_text: str) -> None:
    """
    Generates SSML text from plaintext.
    Given a string of SSML text and an output file name, this function
    calls the Text-to-Speech API. The API returns a synthetic audio
    version of the text, formatted according to the SSML commands. This
    function saves the synthetic audio to the designated output file.

    Args:
        ssml_text: string of SSML text
    """

    # Instantiates a client
    client = texttospeech.TextToSpeechClient()

    # Sets the text input to be synthesized
    synthesis_input = texttospeech.SynthesisInput(ssml=ssml_text)

    # Builds the voice request, selects the language code ("en-US") and
    # the SSML voice gender ("MALE")
    voice = texttospeech.VoiceSelectionParams(
        language_code="en-US", ssml_gender=texttospeech.SsmlVoiceGender.MALE
    )

    # Selects the type of audio file to return
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    # Performs the text-to-speech request on the text input with the selected
    # voice parameters and audio file type
    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    # Writes the synthetic audio to the output file.
    with open("test_example.mp3", "wb") as out:
        out.write(response.audio_content)
        print("Audio content written to file " + "test_example.mp3")
ssml_to_audio = "Hello World!"

In [6]:
import edge_tts


with open("sentence_01.xml", "r") as file:
    text = file.read()

tts = edge_tts.Communicate(text, voice = "hi-IN-MadhurNeural")
await tts.save("test.mp3")

UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 313: character maps to <undefined>